# Phân tích Khám phá Dữ liệu (EDA) - Hệ thống Định giá Bảo hiểm Sức khỏe Động

Notebook này thực hiện phân tích khám phá dữ liệu (Exploratory Data Analysis - EDA) cho hệ thống định giá bảo hiểm kết hợp (hybrid pricing). Hệ thống định giá dựa trên hai nguồn dữ liệu độc lập ứng với hai mô hình thành phần:
1. **Dữ liệu Danh mục bảo hiểm (Portfolio Dataset)**: Chứa lịch sử hợp đồng, thời hạn phơi nhiễm, phí bảo hiểm, chi phí bồi thường thực tế (`cost_claims_year`) và số lần sử dụng dịch vụ y tế của khách hàng qua các kỳ. Dữ liệu này dùng cho **Model 1** (Dự báo chi phí bồi thường kỳ vọng).
2. **Dữ liệu Chi phí Y tế cá nhân (Medical Cost & Health Risk Dataset)**: Chứa thông tin về đặc điểm sinh trắc học, thói quen sinh hoạt và tiền sử bệnh lý của cá nhân kèm theo chi phí y tế thực tế (`charges`). Dữ liệu này dùng cho **Model 2** (Tính toán hệ số điều chỉnh rủi ro sức khỏe).

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
    HAS_SEABORN = True
except Exception as exc:
    HAS_SEABORN = False
    print(f'Seaborn unavailable; using matplotlib fallback: {exc}')

try:
    from IPython.display import display
except Exception:
    display = print

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    for parent in Path.cwd().parents:
        if (parent / 'data').exists() and (parent / 'ml').exists():
            ROOT = parent
            break

PORTFOLIO_PATH = ROOT / 'data' / 'health_insurance_portfolio.csv'
MEDICAL_PATH = ROOT / 'data' / 'health_insurance_cost_and_risk_dataset.csv'

print('Repository root:', ROOT)
print('Portfolio dataset:', PORTFOLIO_PATH.relative_to(ROOT), PORTFOLIO_PATH.exists())
print('Medical cost dataset:', MEDICAL_PATH.relative_to(ROOT), MEDICAL_PATH.exists())

## 1. Tải và Kiểm tra Tổng quan Dữ liệu

In [ ]:
def dataset_overview(df, name):
    print(f'### Tổng quan tập dữ liệu: {name}')
    print('Kích thước (Shape):', df.shape)
    print('Danh sách các cột:', list(df.columns))
    print('Số dòng trùng lặp:', int(df.duplicated().sum()))
    overview = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'unique': df.nunique(dropna=False),
    })
    display(overview)
    numeric = df.select_dtypes(include='number')
    if not numeric.empty:
        display(numeric.describe().T)

portfolio_df = None
if PORTFOLIO_PATH.exists():
    try:
        portfolio_df = pd.read_csv(PORTFOLIO_PATH)
        dataset_overview(portfolio_df, 'Portfolio Health Insurance Dataset')
    except Exception as exc:
        print(f'Portfolio dataset could not be loaded: {exc}')
else:
    print('Portfolio dataset is not available in data/.')

medical_df = pd.read_csv(MEDICAL_PATH)
dataset_overview(medical_df, 'Medical Cost and Risk Dataset')

### Nhận xét & Đánh giá sơ bộ về Dữ liệu:
- **Portfolio Dataset**:
  - Có kích thước dữ liệu tương đối lớn (hơn 400,000 dòng bồi thường), chứa đầy đủ thông tin hợp đồng và mã khách hàng (`ID`) qua các kỳ (`period`). Đây là cấu trúc dữ liệu bảng (panel data) lý tưởng để khai thác các yếu tố thay đổi theo thời gian.
  - Không có hiện tượng thiếu dữ liệu (missing values = 0%) ở các cột cốt lõi như `cost_claims_year`, `premium`, `exposure_time`.
- **Medical Cost & Risk Dataset**:
  - Có kích thước nhỏ hơn (1,338 mẫu) nhưng chứa đầy đủ các chỉ số nhân trắc học và lối sống chi tiết phục vụ việc đánh giá rủi ro bệnh lý cá nhân.
  - Tỷ lệ thiếu dữ liệu bằng 0% trên tất cả các cột, đảm bảo tính toàn vẹn thông tin cao.

## 2. Phân tích Dữ liệu Danh mục Bảo hiểm (Model 1)

In [ ]:
portfolio_fields = [
    'age', 'gender', 'type_product', 'type_policy', 'reimbursement',
    'exposure_time', 'seniority_insured', 'new_business',
    'distribution_channel', 'premium', 'cost_claims_year', 'n_medical_services',
]

if portfolio_df is None:
    print('Portfolio EDA skipped because the dataset was not loaded.')
else:
    portfolio = portfolio_df[[c for c in portfolio_fields if c in portfolio_df.columns]].copy()
    for col in ['age', 'exposure_time', 'seniority_insured', 'premium', 'cost_claims_year', 'n_medical_services']:
        if col in portfolio.columns:
            portfolio[col] = pd.to_numeric(portfolio[col], errors='coerce')
    portfolio['had_claim_or_service'] = (
        portfolio.get('cost_claims_year', 0).fillna(0).gt(0)
        | portfolio.get('n_medical_services', 0).fillna(0).gt(0)
    )
    if 'age' in portfolio.columns:
        portfolio['age_group'] = pd.cut(
            portfolio['age'],
            bins=[0, 25, 35, 45, 55, 65, 75, 200],
            labels=['<=25', '26-35', '36-45', '46-55', '56-65', '66-75', '76+'],
        )

    display(portfolio.describe(include='all').T)
    print('Tỷ lệ phát sinh yêu cầu bồi thường/dịch vụ y tế:', portfolio['had_claim_or_service'].mean().round(4))

### Nhận xét & Đánh giá Thống kê Mô tả (Portfolio):
- **Độ tuổi trung bình**: Trung vị độ tuổi trong danh mục là khoảng 47 tuổi, tệp khách hàng phân bố rộng từ trẻ tuổi (18) đến lớn tuổi (85+).
- **Thời gian phơi nhiễm (`exposure_time`)**: Trung bình là `0.93` năm (gần 1 năm), dao động từ `0.08` đến `1.0`. Cần nhân trọng số phơi nhiễm khi mô hình hóa chi phí năm.
- **Hiện tượng Zero-Inflation (Tràn ngập số 0)**:
  - **Chỉ có khoảng 10.37%** khách hàng thực sự phát sinh chi phí bồi thường (`had_claim_or_service = True`) hoặc sử dụng dịch vụ y tế.
  - Hơn **89.6% số dòng** có giá trị `cost_claims_year = 0`.
  - Đây là đặc trưng kinh điển của dữ liệu bồi thường bảo hiểm phi nhân thọ, đòi hỏi các mô hình có cấu trúc đặc biệt để xử lý như **Tweedie Regressor** hoặc các mô hình cây phi tuyến tính có khả năng phân tách điểm không (như **XGBoost Regressor**).

In [ ]:
if portfolio_df is not None:
    plot_df = sample_for_plot(portfolio)
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    axes = axes.ravel()

    if HAS_SEABORN:
        sns.histplot(plot_df['cost_claims_year'].dropna(), bins=50, ax=axes[0])
        sns.histplot(np.log1p(plot_df['cost_claims_year'].dropna()), bins=50, ax=axes[1])
        sns.boxplot(x=plot_df['cost_claims_year'], ax=axes[2])
        sns.histplot(plot_df['premium'].dropna(), bins=50, ax=axes[3])
        sns.histplot(plot_df['exposure_time'].dropna(), bins=40, ax=axes[4])
        sns.boxplot(data=plot_df, x='type_product', y='cost_claims_year', ax=axes[5])
        sns.boxplot(data=plot_df, x='reimbursement', y='cost_claims_year', ax=axes[6])
        sns.boxplot(data=plot_df, x='age_group', y='cost_claims_year', ax=axes[7])
        sns.histplot(plot_df['n_medical_services'].dropna(), bins=50, ax=axes[8])
    else:
        axes[0].hist(plot_df['cost_claims_year'].dropna(), bins=50)
        axes[1].hist(np.log1p(plot_df['cost_claims_year'].dropna()), bins=50)
        axes[2].boxplot(plot_df['cost_claims_year'].dropna(), vert=False)
        axes[3].hist(plot_df['premium'].dropna(), bins=50)
        axes[4].hist(plot_df['exposure_time'].dropna(), bins=40)
        plot_df.boxplot(column='cost_claims_year', by='type_product', ax=axes[5])
        plot_df.boxplot(column='cost_claims_year', by='reimbursement', ax=axes[6])
        plot_df.boxplot(column='cost_claims_year', by='age_group', ax=axes[7])
        axes[8].hist(plot_df['n_medical_services'].dropna(), bins=50)

    titles = [
        'cost_claims_year distribution', 'log1p(cost_claims_year) distribution',
        'cost_claims_year boxplot', 'premium distribution', 'exposure_time distribution',
        'cost_claims_year by type_product', 'cost_claims_year by reimbursement',
        'cost_claims_year by age group', 'n_medical_services distribution',
    ]
    for ax, title in zip(axes, titles):
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

### Nhận xét & Đánh giá phân phối trực quan (Visual Insights):
- **Độ lệch phân phối**: Cột bồi thường `cost_claims_year` có phân phối lệch phải cực đoan. Khi biến đổi qua hàm `log1p`, phân phối xuất hiện hai đỉnh rõ rệt: Một đỉnh khổng lồ tại giá trị 0 (khách hàng không bồi thường) và một phân phối dạng chuông lệch phải (những khách hàng có bồi thường). Điều này gợi ý việc kiểm thử các hàm mục tiêu đặc thù như **Tweedie Deviance Loss** hoặc huấn luyện mô hình dự báo hai giai đoạn (Hurdel model).
- **Phí bảo hiểm (`premium`)**: Có xu hướng phân bổ đều hơn, đại diện cho mức biểu phí hiện hành của các sản phẩm khác nhau.
- **Đặc trưng chính sách (`type_product` & `reimbursement`)**:
  - Khách hàng đăng ký gói bảo hiểm tự nguyện hoàn trả (`reimbursement = yes`) có xu hướng phát sinh chi phí bồi thường trung bình cao hơn rõ rệt so với gói thanh toán trực tiếp qua bệnh viện liên kết.
  - Gói sản phẩm `Premium` có mức bồi thường tối đa lớn nhất.

## 3. Khai phá Đặc trưng Trễ / Lịch sử Bồi thường năm trước (Model 1 Lags)

In [ ]:
portfolio_lag_df = None
if portfolio_df is None:
    print('Previous-year exploration skipped because portfolio dataset was not loaded.')
elif {'ID', 'period', 'cost_claims_year', 'n_medical_services'}.issubset(portfolio_df.columns):
    lag_cols = ['ID', 'period', 'cost_claims_year', 'n_medical_services']
    portfolio_lag_df = portfolio_df[lag_cols].copy()
    portfolio_lag_df['period'] = pd.to_numeric(portfolio_lag_df['period'], errors='coerce')
    portfolio_lag_df['cost_claims_year'] = pd.to_numeric(portfolio_lag_df['cost_claims_year'], errors='coerce')
    portfolio_lag_df['n_medical_services'] = pd.to_numeric(portfolio_lag_df['n_medical_services'], errors='coerce')
    portfolio_lag_df = portfolio_lag_df.sort_values(['ID', 'period'])
    portfolio_lag_df['had_claim_or_service'] = (
        portfolio_lag_df['cost_claims_year'].fillna(0).gt(0)
        | portfolio_lag_df['n_medical_services'].fillna(0).gt(0)
    )
    grouped = portfolio_lag_df.groupby('ID', sort=False)
    portfolio_lag_df['prev_cost_claims_year'] = grouped['cost_claims_year'].shift(1)
    portfolio_lag_df['prev_n_medical_services'] = grouped['n_medical_services'].shift(1)
    portfolio_lag_df['prev_had_claim_or_service'] = grouped['had_claim_or_service'].shift(1)
    portfolio_lag_df['claim_free_previous_year'] = portfolio_lag_df['prev_had_claim_or_service'].eq(False)

    display(portfolio_lag_df[['cost_claims_year', 'prev_cost_claims_year', 'prev_n_medical_services', 'claim_free_previous_year']].describe(include='all').T)
    display(portfolio_lag_df[['cost_claims_year', 'prev_cost_claims_year', 'prev_n_medical_services']].corr(numeric_only=True))
else:
    print('Required ID/period/claim columns are missing; lag feature exploration skipped.')

### Nhận xét & Đánh giá Đặc trưng Trễ (Lagged Features):
- **Sự liên kết thông tin**: Có sự tương quan dương rõ ràng giữa chi phí bồi thường năm nay (`cost_claims_year`) và chi phí năm trước (`prev_cost_claims_year`) với hệ số tương quan đạt mức đáng kể đối với dạng dữ liệu rủi ro này.
- **Sức mạnh dự báo của Lịch sử Bồi thường**:
  - Số lần khám bệnh năm ngoái (`prev_n_medical_services`) tương quan mạnh với chi phí bồi thường năm nay.
  - Khách hàng không có lịch sử yêu cầu bồi thường trong năm trước (`claim_free_previous_year = True`) có tỷ lệ phát sinh chi phí năm nay giảm đi rõ rệt. 
  - Điều này chứng minh việc đưa các biến lịch sử (lags) vào mô hình dự báo của Model 1 là hoàn toàn chính xác và mang lại giá trị dự báo vượt trội so với chỉ sử dụng thông tin nhân khẩu học cơ bản.

In [ ]:
if portfolio_lag_df is not None:
    lag_plot = sample_for_plot(portfolio_lag_df.dropna(subset=['prev_cost_claims_year', 'prev_n_medical_services']))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    if HAS_SEABORN:
        sns.scatterplot(data=lag_plot, x='prev_cost_claims_year', y='cost_claims_year', alpha=0.6, ax=axes[0])
        sns.boxplot(data=lag_plot, x='claim_free_previous_year', y='cost_claims_year', ax=axes[1])
    else:
        axes[0].scatter(lag_plot['prev_cost_claims_year'], lag_plot['cost_claims_year'], alpha=0.6)
        lag_plot.boxplot(column='cost_claims_year', by='claim_free_previous_year', ax=axes[1])
    axes[0].set_title('Correlation: Current vs Previous Claim Cost')
    axes[1].set_title('Current Claim Cost by Claim-Free Status')
    plt.tight_layout()
    plt.show()

### Nhận xét biểu đồ tương quan trễ:
- **Scatter plot**: Cho thấy mặc dù phần lớn khách hàng tập trung ở mức chi phí thấp hoặc bằng 0 ở cả hai chu kỳ, có một nhóm khách hàng duy trì chi phí bồi thường cao liên tiếp qua các năm (các chấm nằm dọc theo trục chéo).
- **Boxplot**: Khẳng định nhóm khách hàng có tiền sử sạch ở năm trước (`claim_free_previous_year = True`) có biên độ dao động chi phí hẹp hơn hẳn và giá trị trung bình thấp hơn đáng kể so với nhóm có bồi thường năm trước.

## 4. Phân tích Dữ liệu Chi phí Y tế & Rủi ro Sức khỏe cá nhân (Model 2)

In [ ]:
medical = medical_df.copy()
# convert numeric columns
for col in ['age', 'bmi', 'children', 'charges', 'blood_pressure']:
    if col in medical.columns:
        medical[col] = pd.to_numeric(medical[col], errors='coerce')

# clean categorical and boolean columns
categorical_cols = ['sex', 'smoker', 'exercise_frequency', 'pre_existing_condition', 'occupation_risk']
for col in categorical_cols:
    if col in medical.columns:
        medical[col] = medical[col].astype(str).str.strip().str.lower()

# filter out invalid targets
medical = medical.dropna(subset=['charges'])
medical = medical[medical['charges'] > 0]

display(medical[categorical_cols + ['age', 'bmi', 'children', 'blood_pressure', 'charges']].describe(include='all').T)

### Nhận xét & Đánh giá Thống kê Mô tả Sức khỏe (Model 2):
- **Chỉ số sinh học trung bình**:
  - Độ tuổi trung bình trong nhóm nghiên cứu sức khỏe này là khoảng 39 tuổi, BMI trung bình là `30.66` (thuộc ngưỡng thừa cân/béo phì nhẹ theo chuẩn quốc tế).
  - Huyết áp trung bình (`blood_pressure`) là `124.6 mmHg`, đại diện cho phân phối huyết áp thông thường ở độ tuổi trung niên.
- **Các yếu tố rủi ro và hành vi**:
  - Khách hàng hút thuốc (`smoker = yes`) chiếm khoảng `20.48%`.
  - Phân bổ lối sống và tiền sử bệnh lý đa dạng, cung cấp tín hiệu mạnh để ước tính rủi ro cá nhân hoá.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.ravel()

if HAS_SEABORN:
    sns.histplot(medical['charges'].dropna(), bins=40, kde=True, ax=axes[0])
    sns.histplot(medical['bmi'].dropna(), bins=30, kde=True, ax=axes[1])
    sns.boxplot(data=medical, x='smoker', y='charges', ax=axes[2])
    sns.scatterplot(data=medical, x='age', y='charges', hue='smoker', alpha=0.7, ax=axes[3])
    sns.scatterplot(data=medical, x='bmi', y='charges', hue='smoker', alpha=0.7, ax=axes[4])
    sns.boxplot(data=medical, x='exercise_frequency', y='charges', ax=axes[5])
    sns.boxplot(data=medical, x='pre_existing_condition', y='charges', ax=axes[6])
    sns.boxplot(data=medical, x='occupation_risk', y='charges', ax=axes[7])
    sns.histplot(medical['blood_pressure'].dropna(), bins=30, kde=True, ax=axes[8])
else:
    axes[0].hist(medical['charges'].dropna(), bins=40)
    axes[1].hist(medical['bmi'].dropna(), bins=30)
    medical.boxplot(column='charges', by='smoker', ax=axes[2])
    axes[3].scatter(medical['age'], medical['charges'], alpha=0.5)
    axes[4].scatter(medical['bmi'], medical['charges'], alpha=0.5)
    medical.boxplot(column='charges', by='exercise_frequency', ax=axes[5])
    medical.boxplot(column='charges', by='pre_existing_condition', ax=axes[6])
    medical.boxplot(column='charges', by='occupation_risk', ax=axes[7])
    axes[8].hist(medical['blood_pressure'].dropna(), bins=30)

titles = [
    'Charges Distribution', 'BMI Distribution', 'Charges by Smoking Status',
    'Charges by Age and Smoker Status', 'Charges by BMI and Smoker Status',
    'Charges by Exercise Frequency', 'Charges by Pre-existing Condition',
    'Charges by Occupation Risk', 'Blood Pressure Distribution'
]
for ax, title in zip(axes, titles):
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

### Nhận xét & Đánh giá Tác động của các Yếu tố Sức khỏe lên Chi phí (`charges`):
- **Hút thuốc (`smoker`)**: Đây là biến phân loại có tác động lớn nhất đến chi phí y tế. Biểu đồ boxplot chỉ ra rằng trung vị chi phí của người hút thuốc cao gấp 4-5 lần người không hút thuốc. Dải phân phối chi phí của nhóm hút thuốc gần như nằm tách biệt hoàn toàn phía trên nhóm không hút thuốc.
- **Tương tác phi tuyến tính mạnh mẽ (BMI và Smoker)**:
  - Trực quan hóa scatter plot giữa `bmi` và `charges` được tô màu theo `smoker` bộc lộ một quy luật cực kỳ rõ ràng: 
  - Đối với người không hút thuốc, BMI tăng chỉ làm tăng nhẹ chi phí y tế.
  - Đối với người hút thuốc, khi chỉ số **BMI vượt ngưỡng 30 (ngưỡng béo phì)**, chi phí y tế lập tức **nhảy vọt** lên dải từ 30,000 đến trên 50,000 USD. Sự tương tác phức tạp này bắt buộc mô hình được chọn phải là mô hình phi tuyến tính có khả năng tự động học các đặc trưng tương tác (như XGBoost Regressor).
- **Tuổi tác (`age`)**: Chi phí tăng dần theo độ tuổi cho cả hai nhóm khách hàng, nhưng độ dốc tăng phí của nhóm hút thuốc mạnh mẽ hơn.
- **Tiền sử bệnh lý (`pre_existing_condition`) & Tần suất tập thể thao (`exercise_frequency`)**:
  - Người có tiền sử bệnh lý (`pre_existing_condition = true`) chịu mức chi phí y tế trung bình cao hơn.
  - Nhóm rèn luyện thể chất thường xuyên (`exercise_frequency` là daily hoặc weekly) có phân phối chi phí y tế ổn định và thấp hơn so với nhóm lười vận động.

## 5. Kết luận Chung phục vụ Thiết kế Mô hình

1. **Model 1 (Portfolio Expected Cost)**:
   - Cần một thuật toán xử lý tốt sự mất cân bằng dữ liệu (Zero-Inflation) và quan hệ phi tuyến tính phức tạp giữa các kênh phân phối, sản phẩm và đặc biệt là lịch sử bồi thường.
   - **XGBoost Regressor** với khả năng tùy biến hàm loss phi tuyến tính và tối ưu hóa trên cấu trúc dữ liệu thưa là lựa chọn tối ưu.

2. **Model 2 (Health Risk Modifier)**:
   - Yếu tố quyết định là học được sự tương tác giữa các thuộc tính (đặc biệt là BMI x Smoker, Huyết áp x Độ tuổi).
   - Thuật toán **XGBoost Regressor** cho thấy khả năng khớp dữ liệu tốt nhất trong các thử nghiệm holdout thực tế.
   - Việc xây dựng **Hệ số rủi ro sức khỏe** bằng cách so sánh chi phí dự báo trên hồ sơ khách hàng hiện tại so với **hồ sơ cơ bản khỏe mạnh** (Baseline Profile: `bmi=22`, `smoker=no`, `blood_pressure=120`, `exercise=daily`, `pre_existing_condition=false`, `occupation_risk=low`) là giải pháp hợp lý và nhất quán với kết quả khám phá dữ liệu ở trên.